## GPU CHECK

In [1]:
!nvidia-smi
!cat /proc/driver/nvidia/version

# Which libcuda candidates are on your system?
!ldconfig -p | grep libcuda.so.1

# Is a conda/venv path shadowing libcuda?
!echo "$LD_LIBRARY_PATH"
!find ~/miniconda3 -name "libcuda.so*" 2>/dev/null | head
!find ~/stroke_cleaned/.venv -name "libcuda.so*" 2>/dev/null | head


Tue Sep 23 10:04:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.181                Driver Version: 570.181        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:41:00.0 Off |                  Off |
|  0%   34C    P8             30W /  480W |      15MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Re-Define the Model to memory ... must run!

In [3]:

"""
Stroke Lesion Segmentation v 1.2

This training script builds upon prior production and cropped variants but adds
support for arbitrary volumetric input sizes.  It automatically determines
the largest spatial dimensions present in a dataset and pads smaller volumes
so that all inputs share the same shape.  The script retains detailed
logging, memory monitoring, data augmentation and custom layers from the
previous versions while incorporating recommendations from the latest model
evaluation:

* Dice/boundary loss weights adjusted to emphasise boundary precision
* Over‑segmentation mitigation via adjustable decision threshold
* Slightly stronger augmentation (rotations/flips/gamma) when overfitting
  is suspected
* Tunable L2 regularisation and dropout rates
* Longer warm‑up and lower minimum learning rate

The batch size is fixed at 2, but input volumes may be of any shape as long
as the corresponding mask has identical dimensions.  All three spatial
dimensions are padded to the maximum observed size for consistent training.
"""

from logging import config
import os
import sys
import logging
from pathlib import Path

# ---- Environment (set BEFORE importing TensorFlow) ----
import os


# Keep: quiet logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Optional: better GPU allocator (helps reduce fragmentation on long runs)
# Works with TF 2.10+ built for CUDA 11/12.
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Don't set for normal training:
# - CUDA_LAUNCH_BLOCKING=1  # debug-only; forces sync and can make training very slow
# - TF_XLA_FLAGS / XLA_FLAGS  # generally unnecessary on TF 2.20; can cause confusion
# - TF_ENABLE_ONEDNN_OPTS=0  # controls CPU-only kernels; leave default unless you need bit-for-bit CPU numerics


import tensorflow as tf

# See GPUs and enable memory growth (good practice)
gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"Could not set memory growth on {gpu}: {e}")

# Optional: use all visible GPUs
strategy = tf.distribute.MirroredStrategy() if gpus else tf.distribute.get_strategy()
print("Strategy:", type(strategy).__name__)


Path("logs").mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
# Use two file handlers and one stream handler.  One file captures all
# high‑level events (INFO and above) and the other captures per‑process
# debugging output.  A console stream is kept for quick feedback.

Path("logs").mkdir(parents=True, exist_ok=True)


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/smart_sota_dynamic.log'),
        logging.FileHandler(f'logs/training_dynamic_{os.getpid()}.debug.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('SmartSOTA_Dynamic')
logger.setLevel(logging.DEBUG)


# Build/compile inside the scope if you use strategy
# with strategy.scope():
#     model = ...
#     model.compile(...)

import sys, faulthandler, signal
faulthandler.enable()

def _log_excepthook(exc_type, exc, tb):
    logger.exception("💥 Uncaught exception", exc_info=(exc_type, exc, tb))

sys.excepthook = _log_excepthook
faulthandler.register(signal.SIGTERM, all_threads=True, chain=True)

# ---------------------------------------------------------------------------
# Imports with graceful degradation
# ---------------------------------------------------------------------------
try:
    import tensorflow as tf
    from tensorflow.keras import layers
   

    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_bfloat16")  # reduces activation memory ~2x on modern GPUs
# (we already cast to float32 inside your losses/metrics, so this is safe)

    import numpy as np
    import nibabel as nib
    from scipy.ndimage import zoom, binary_dilation, rotate
    from sklearn.model_selection import KFold
    from skimage import measure
    import json
    import time
    import math
    import random
    import gc
    import psutil
    from functools import lru_cache
    from concurrent.futures import ThreadPoolExecutor
    logger.info("✅ All imports successful")
except ImportError as e:
    logger.critical(f"❌ Import failed: {e}")
    sys.exit(1)

# ---------------------------------------------------------------------------
# Runtime configuration
# ---------------------------------------------------------------------------
# Disable eager execution for performance and to avoid certain CUDNN issues.
tf.config.run_functions_eagerly(False)
logger.info(f"TensorFlow eager execution: {tf.executing_eagerly()}")

warnings_to_ignore = [UserWarning, DeprecationWarning, FutureWarning]
for w in warnings_to_ignore:
    tf.autograph.set_verbosity(0)
import warnings
for w in warnings_to_ignore:
    warnings.filterwarnings("ignore", category=w)
warnings.filterwarnings("ignore", module="nibabel")

logger.info(
    f"Environment verified:\n"
    f"- Python {sys.version}\n"
    f"- TensorFlow {tf.__version__}\n"
    f"- NumPy {np.__version__}\n"
    f"- GPU devices: {len(tf.config.list_physical_devices('GPU'))}"
)

# ---------------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------------
class DynamicTrainingConfig:
    """Configuration class supporting variable input shapes.

    The first call to `detect_input_shape` will populate INPUT_SHAPE based on
    observed dataset maxima.  All other hyperparameters may be tuned from
    outside to reflect recommendations from prior evaluations.
    """

    # Root directory containing Images and Masks (subdirectories or mixed)
    DATA_DIR: Path = Path("/home/rbielski/Atlas_2/Training_Split/Training_Set")

    # The input shape will be set by detect_input_shape; default to None
    INPUT_SHAPE = None

    # Data split configuration
    VALIDATION_SPLIT = 0.15
    SMALL_LESION_THRESHOLD = 100

    # Training schedule
    BATCH_SIZE = 2
    INITIAL_EPOCH = 0
    TOTAL_EPOCHS = 200
    INITIAL_LR = 1e-4
    MIN_LR = 5e-7               # Lower minimum LR for potential late training plateaus
    WARMUP_EPOCHS = 15          # Extended warmup
    MAX_GRAD_NORM = 1.0

    # Model architecture
    BASE_FILTERS = 8
    DROPOUT_RATE = 0.55         # Balanced dropout for robustness
    L2_REG = 1.5e-3             # Moderate L2 regularisation
    MAMBA_DEPTH = 2
    SAM_HEADS = 4

    # Data augmentation
    AUGMENTATION_INTENSITY = 0.5
    SYNTHETIC_LESION_PROB = 0.3
    ROTATION_RANGE = 20         # Slightly larger range than original

    # Loss configuration (based on evaluation recommendations)
    USE_BOUNDARY_LOSS = True
    DICE_LOSS_WEIGHT = 0.4
    BOUNDARY_LOSS_WEIGHT = 0.6
    DEEP_SUPERVISION_WEIGHTS = [0.1, 0.2, 0.3]
    
    #Tweaks
    # Add these defaults anywhere among the other hyperparams:
    DICE_WEIGHT: float = 0.4
    BOUNDARY_WEIGHT: float = 0.6
    # inside class DynamicTrainingConfig:
    RESAMPLE_TO_TARGET = True   # resample both image & mask to INPUT_SHAPE[:-1]


    # Decision threshold for converting logits to binary masks
    DECISION_THRESHOLD = 0.5    # Raise threshold to mitigate over‑segmentation

    # Output directories
    MODEL_DIR = Path("models/dynamic_production")
    CALLBACKS_DIR = Path("callbacks/dynamic_production")

    def __init__(self):
        # Timestamp for saving model checkpoints
        self.timestamp = time.strftime("%Y%m%d_%H%M%S")
        self.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        self.CALLBACKS_DIR.mkdir(parents=True, exist_ok=True)
        # Write config to JSON for reproducibility
        config_dict = {k: v for k, v in self.__dict__.items()
                      if not k.startswith('__') and not callable(v)}
        with open(self.MODEL_DIR / "config.json", "w") as f:
            json.dump(config_dict, f, indent=2, default=str)
        logger.info(
            f"📋 DynamicTrainingConfig initialised:\n"
            f"   Data directory: {self.DATA_DIR}\n"
            f"   Batch size: {self.BATCH_SIZE}\n"
            f"   Warmup epochs: {self.WARMUP_EPOCHS}\n"
            f"   Minimum LR: {self.MIN_LR}\n"
            f"   Dice/boundary weights: {self.DICE_LOSS_WEIGHT}:{self.BOUNDARY_LOSS_WEIGHT}\n"
        )

    @property
    def model_path(self) -> Path:
        return self.MODEL_DIR / f"smart_sota_dynamic_{self.timestamp}.keras"

    
    @property
    def checkpoint_path(self) -> Path:
        return self.CALLBACKS_DIR / "best_model_dynamic.weights.h5"



# ---------------------------------------------------------------------------
# Custom layers (identical to previous implementations)
# ---------------------------------------------------------------------------
# put this once near the top (same place you imported for losses)
try:
    from keras.saving import register_keras_serializable
except Exception:
    from tensorflow.keras.utils import register_keras_serializable  # fallback


@register_keras_serializable(package="custom")
class ResidualConvBlock(layers.Layer):
    """Residual block using LayerNorm (more stable than BN for very small batches)."""
    def __init__(self, filters, kernel_reg=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_reg = kernel_reg

    def build(self, input_shape):
        self.conv1 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.ln1 = layers.LayerNormalization(epsilon=1e-5)
        self.conv2 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.ln2 = layers.LayerNormalization(epsilon=1e-5)
        self.dropout = layers.SpatialDropout3D(0.1)
        self.residual_conv = layers.Conv3D(self.filters, 1, padding='same')
        self.residual_ln = layers.LayerNormalization(epsilon=1e-5)
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.conv1(inputs)
        x = self.ln1(x)
        x = tf.nn.relu(x)
        x = self.dropout(x, training=training)
        x = self.conv2(x)
        x = self.ln2(x)
        residual = self.residual_conv(inputs)
        residual = self.residual_ln(residual)
        return tf.nn.relu(x + residual)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_reg": tf.keras.regularizers.serialize(self.kernel_reg)
                           if self.kernel_reg else None
        })
        return config


@register_keras_serializable(package="custom")
class VisionMambaBlock(layers.Layer):
    """Efficient vision Mamba block with dynamic input support"""
    def __init__(self, filters, kernel_size=3, expansion=2, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.expansion = expansion

    def build(self, input_shape):
        self.in_conv = layers.Conv3D(
            self.filters * self.expansion,
            1,
            use_bias=False,
            padding='same',
            data_format='channels_last')
        self.spatial_conv = layers.Conv3D(
            self.filters * self.expansion,
            self.kernel_size,
            padding='same',
            use_bias=False,
            data_format='channels_last')
        self.out_conv = layers.Conv3D(
            self.filters, 1,
            padding='same',
            data_format='channels_last')
        self.norm = layers.LayerNormalization()
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.in_conv(inputs)
        x = tf.nn.relu(x)
        x = self.spatial_conv(x)
        x = tf.cast(tf.nn.relu(x), inputs.dtype)
        x = self.dropout(x, training=training)
        x = self.out_conv(x)
        x = self.norm(x)
        return x + inputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "expansion": self.expansion
        })
        return config

@register_keras_serializable(package="custom")
class SAM2Attention(layers.Layer):
    """Enhanced SAM2 attention with hierarchical memory banks"""
    def __init__(self, filters, heads, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.heads = heads
        self.depth = filters // heads
        if filters % heads != 0:
            raise ValueError("Filters must be divisible by heads")

    def build(self, input_shape):
        self.query = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.key = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.value = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.out_conv = layers.Conv3D(input_shape[-1], 1, data_format='channels_last')
        self.memory_bank = self.add_weight(
            name='memory_bank',
            shape=(1, 1, 1, 1, self.filters),
            initializer='zeros',
            trainable=True
        )
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        depth_dim = tf.shape(inputs)[3]
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)
        k = k + self.memory_bank
        v = v + self.memory_bank
        q = self._split_heads_safe(q, batch_size, height, width, depth_dim)
        k = self._split_heads_safe(k, batch_size, height, width, depth_dim)
        v = self._split_heads_safe(v, batch_size, height, width, depth_dim)
        dk = tf.cast(self.depth, q.dtype)
        attn_logits = tf.matmul(q, k, transpose_b=True)
        attn_logits = attn_logits / tf.math.sqrt(dk)
        attn_weights = tf.nn.softmax(attn_logits, axis=-1)
        attn_output = tf.matmul(attn_weights, v)
        attn_output = self._combine_heads_safe(attn_output, batch_size, height, width, depth_dim)
        output = self.out_conv(attn_output)
        output = self.dropout(output, training=training)
        return output + inputs

    def _split_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.reshape(x, [batch_size, height, width, depth_dim, self.heads, self.depth])
        return tf.transpose(x, perm=[0, 4, 1, 2, 3, 5])

    def _combine_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.transpose(x, perm=[0, 2, 3, 4, 1, 5])
        return tf.reshape(x, [batch_size, height, width, depth_dim, self.filters])

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "heads": self.heads
        })
        return config

# ---------------------------------------------------------------------------
# Build the segmentation model (UNet-like with your custom blocks)
# ---------------------------------------------------------------------------
def build_dynamic_model(config: DynamicTrainingConfig) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=config.INPUT_SHAPE)  # (D,H,W,1)

    x = inputs
    skips = []
    filters = config.BASE_FILTERS

    # Encoder
    for _ in range(4):
        x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
        x = VisionMambaBlock(filters)(x)
        skips.append(x)
        x = layers.MaxPool3D(pool_size=2)(x)
        filters *= 2

    # Bottleneck
    x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
    x = SAM2Attention(filters, heads=config.SAM_HEADS)(x)

    # Decoder
    for d in reversed(range(4)):
        filters //= 2
        x = layers.UpSampling3D(size=2)(x)
        x = layers.Concatenate()([x, skips[d]])
        x = ResidualConvBlock(filters, kernel_reg=tf.keras.regularizers.l2(config.L2_REG))(x)
        x = VisionMambaBlock(filters)(x)

    # IMPORTANT: output logits (no activation). Dice in your loss applies sigmoid.
    # Build model – replace the head
    outputs = layers.Conv3D(1, kernel_size=1, activation="sigmoid", name="probs")(x)


    return tf.keras.Model(inputs=inputs, outputs=outputs, name="SmartSOTA_Dynamic")



# ---------------------------------------------------------------------------
# Utility functions for memory monitoring
# ---------------------------------------------------------------------------
def log_memory_usage(stage: str) -> None:
    process = psutil.Process(os.getpid())
    gb_used = process.memory_info().rss / 1024**3
    gpu_mem = []
    try:
        for i in range(4):
            alloc = tf.config.experimental.get_memory_info(f'GPU:{i}')
            gpu_mem.append(f"GPU{i}: {alloc['current']/1e9:.2f}GB")
    except Exception:
        gpu_mem = ["GPU mem tracking failed"]
    try:
        disk_usage = psutil.disk_usage('/')
        disk_free_gb = disk_usage.free / 1024**3
        disk_info = f"Disk: {disk_free_gb:.1f}GB free"
    except Exception:
        disk_info = "Disk: unavailable"
    logger.info(f"Memory at {stage}: CPU={gb_used:.2f}GB | {' | '.join(gpu_mem)} | {disk_info}")


# ---------------------------------------------------------------------------
# Dataset inspection and loading
# ---------------------------------------------------------------------------
from pathlib import Path
import numpy as np
import nibabel as nib
import gc



def detect_input_shape(data_dir: Path) -> tuple:
    """
    Determine the maximum spatial (D,H,W) across NIfTI volumes under `data_dir`,
    then round each dimension UP to the nearest multiple of 16.

    We consider any .nii.gz with at least 3 dims. If none are valid, an error is raised.
    Logs fall back to print() if a global `logger` isn't available.
    """
    import math
    import nibabel as nib

    log = globals().get("logger", None)
    def _info(msg: str):
        if log is not None:
            log.info(msg)
        else:
            print(msg)

    _info("🔍 Detecting input shape from dataset…")

    # Scan all NIfTI files under the root (Images/Masks are fine; we only read headers/shapes)
    image_files = list(data_dir.rglob("*.nii.gz"))
    max_shape = [0, 0, 0]
    invalid = []

    if not image_files:
        raise FileNotFoundError(f"No .nii.gz files found under {data_dir}")

    for f in image_files:
        try:
            img = nib.load(str(f))
            shp = img.shape
            # Need at least 3 spatial dims
            if len(shp) >= 3:
                for i in range(3):
                    max_shape[i] = max(max_shape[i], int(shp[i]))
            else:
                invalid.append(f"{f.name}: shape {shp} has fewer than 3 dims")
        except Exception as e:
            invalid.append(f"{f.name}: failed to load ({e})")

    if all(dim == 0 for dim in max_shape):
        details = ("Issues encountered:\n  - " + "\n  - ".join(invalid)) if invalid else "No details."
        raise RuntimeError(f"No valid 3-D NIfTI files found in {data_dir}. {details}")

    def _ceil16(x: int) -> int:
        return int(math.ceil(x / 16.0) * 16)

    rounded_shape = tuple(_ceil16(dim) for dim in max_shape)

    _info(
        f"📐 Detected max volume dimensions: {tuple(max_shape)} → "
        f"rounded up to: {rounded_shape}"
    )
    return rounded_shape


# --- Replace your existing load_generic_dataset with this version ---
import gc
import numpy as np
import nibabel as nib
from pathlib import Path
import re

def load_generic_dataset(config: DynamicTrainingConfig):
    """
    Loads pairs from:
      <config.DATA_DIR>/Images/*T1w*.nii.gz
      <config.DATA_DIR>/Masks/*mask*.nii.gz

    Pairing:
      image key: strip '_T1w' (before .nii.gz)
      mask  key: strip '_label-..._desc-..._mask' (or trailing '_mask')
    Returns:
      pairs: list[(image_path, mask_path)]
      lesion_presence: np.array of {0,1} per pair (mask has any > 0)
    """
    logger.info("📚 Loading generic dataset (RB pairing rules)...")
    log_memory_usage("dataset_load_start")

    base = config.DATA_DIR
    images_dir = (base / "Images")
    masks_dir  = (base / "Masks")
    if not images_dir.exists() or not masks_dir.exists():
        raise FileNotFoundError(f"Expected subfolders 'Images' and 'Masks' under {base}")

    def img_key(p: Path) -> str:
        s = p.name[:-7] if p.name.endswith(".nii.gz") else p.name
        return s.replace("_T1w", "")

    def msk_key(p: Path) -> str:
        s = p.name[:-7] if p.name.endswith(".nii.gz") else p.name
        s2 = re.sub(r"_label-[^_]+_desc-[^_]+_mask$", "", s)
        if s2 == s:
            s2 = re.sub(r"_mask$", "", s2)
        return s2

    images = sorted([p for p in images_dir.glob("*.nii.gz") if "T1w" in p.name and "mask" not in p.name])
    masks  = sorted([p for p in masks_dir.glob("*.nii.gz")  if "mask" in p.name])

    logger.info(f"✅ Found {len(images)} images under {images_dir}")
    logger.info(f"✅ Found {len(masks)} masks under  {masks_dir}")

    img_map = {img_key(p): p for p in images}
    msk_map = {msk_key(p): p for p in masks}
    keys = sorted(set(img_map).intersection(msk_map.keys()))

    if not keys:
        # Print a few sample names/keys to explain WHY zero pairs
        some_imgs = list(img_map.items())[:5]
        some_msks = list(msk_map.items())[:5]
        logger.error("No image–mask pairs matched. Example keys (image -> file):")
        for k, v in some_imgs:
            logger.error(f"  {k} -> {v.name}")
        logger.error("Example keys (mask -> file):")
        for k, v in some_msks:
            logger.error(f"  {k} -> {v.name}")
        raise RuntimeError("No pairs matched. Check filename patterns / key rules above.")

    pairs = []
    lesion_counts = []
    for k in keys:
        img_p = img_map[k]
        msk_p = msk_map[k]
        try:
            mask_obj = nib.load(str(msk_p))
            has_lesion = bool(np.any(mask_obj.get_fdata() > 0))
            lesion_counts.append(1 if has_lesion else 0)
            pairs.append((img_p, msk_p))
        except Exception as e:
            logger.warning(f"Skipping pair for {k}: {e}")
        finally:
            try:
                del mask_obj
            except:
                pass
            gc.collect()

    logger.info(f"📊 Created {len(pairs)} image–mask pairs")
    if lesion_counts:
        logger.info(f"🧠 Lesion presence: {np.mean(lesion_counts)*100:.2f}%")
    log_memory_usage("dataset_load_end")
    return pairs, np.array(lesion_counts, dtype=np.int32)


def create_stratified_splits(pairs, lesion_presence, batch_size, test_size=0.1):
    """Generate train/validation splits compatible with batch size.

    We adapt the original function to ensure that each split contains a number
    of samples divisible by the batch size.  Stratification is performed
    based on lesion presence.
    """
    total_samples = len(pairs)
    test_samples = math.floor(total_samples * test_size)
    train_samples = total_samples - test_samples
    # Round down to nearest batch size
    test_samples = (test_samples // batch_size) * batch_size
    train_samples = total_samples - test_samples
    # Guarantee at least one batch in each split
    if test_samples < batch_size:
        test_samples = batch_size
        train_samples = total_samples - test_samples
    if train_samples < batch_size:
        train_samples = batch_size
        test_samples = total_samples - train_samples
    logger.info(
        f"🧮 Dataset split: Train={train_samples}"
        f" ({train_samples/total_samples*100:.1f}%), "
        f"Validation={test_samples} ({test_samples/total_samples*100:.1f}%)"
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, test_idx in kf.split(pairs, lesion_presence):
        if len(test_idx) >= test_samples:
            test_idx = test_idx[:test_samples]
            break
    train_pairs = [pairs[i] for i in train_idx]
    test_pairs = [pairs[i] for i in test_idx]
    train_lesions = np.mean([lesion_presence[i] for i in train_idx])
    test_lesions = np.mean([lesion_presence[i] for i in test_idx])
    logger.info(
        f"⚖️ Lesion representation: Train={train_lesions*100:.1f}%, "
        f"Validation={test_lesions*100:.1f}%"
    )
    return train_pairs, test_pairs

def pad_and_center_crop(volume: np.ndarray, target_shape: tuple) -> np.ndarray:
    """
    Symmetrically pad (if smaller) or center-crop (if larger) a 3D volume to target_shape.
    Works for both images (float) and masks (binary/float). No interpolation is used.
    """
    assert volume.ndim == 3, f"Expected 3D volume, got {volume.ndim}D"
    z, y, x = volume.shape
    tz, ty, tx = target_shape
    out = volume

    # Center-crop if needed
    if z > tz:
        start = (z - tz) // 2
        out = out[start:start+tz, :, :]
        z = tz
    if y > ty:
        start = (y - ty) // 2
        out = out[:, start:start+ty, :]
        y = ty
    if x > tx:
        start = (x - tx) // 2
        out = out[:, :, start:start+tx]
        x = tx

    # Symmetric pad if needed
    pad_z = max(0, tz - z)
    pad_y = max(0, ty - y)
    pad_x = max(0, tx - x)
    if pad_z or pad_y or pad_x:
        pz0, pz1 = pad_z // 2, pad_z - pad_z // 2
        py0, py1 = pad_y // 2, pad_y - pad_y // 2
        px0, px1 = pad_x // 2, pad_x - pad_x // 2
        out = np.pad(out, ((pz0, pz1), (py0, py1), (px0, px1)), mode="constant", constant_values=0)
    return out

# --- Center-slice helpers (shared crop/pad for image & mask) -----------------
def compute_center_slices(in_shape, out_shape):
    """
    Return input slices that pick the centered sub-volume when cropping, or the
    full axis when padding. Use these slices for BOTH image and mask.
    """
    inD, inH, inW = in_shape
    outD, outH, outW = out_shape
    slices = []
    for i_len, o_len in zip((inD, inH, inW), (outD, outH, outW)):
        if i_len >= o_len:
            start = (i_len - o_len) // 2
            end   = start + o_len
            slices.append(slice(start, end))
        else:
            # padding case: take the whole input on that axis
            slices.append(slice(0, i_len))
    return tuple(slices)  # (sd, sh, sw)

def apply_center_crop_or_pad(vol, in_slices, out_shape):
    """
    Apply the provided input slices, then center-pad into out_shape.
    Use the SAME in_slices for image and mask to guarantee identical transform.
    """
    sub = vol[in_slices[0], in_slices[1], in_slices[2]]
    out = np.zeros(out_shape, dtype=vol.dtype)
    # center place the 'sub' into out
    offs = tuple((o - s) // 2 for s, o in zip(sub.shape, out_shape))
    out[
        offs[0]:offs[0]+sub.shape[0],
        offs[1]:offs[1]+sub.shape[1],
        offs[2]:offs[2]+sub.shape[2],
    ] = sub
    return out.astype(np.float32)


# ---------------------------------------------------------------------------
# Data generator (no augmentations). Only resampling (optional) + center crop/pad.
# ---------------------------------------------------------------------------
import gc
from functools import lru_cache

import nibabel as nib
import numpy as np
import psutil
from scipy.ndimage import zoom
import tensorflow as tf

@lru_cache(maxsize=128)
def _load_vol_canonical(path: str) -> np.ndarray:
    """Load NIfTI as RAS-canonical and return float32 array."""
    img = nib.load(path)
    img = nib.as_closest_canonical(img)  # standardize orientation
    return img.get_fdata().astype(np.float32)

def _load_image(path: str) -> np.ndarray:
    return _load_vol_canonical(path)

def _load_mask_bin(path: str) -> np.ndarray:
    return (_load_vol_canonical(path) > 0.5).astype(np.float32)

def _center_crop_or_pad(vol: np.ndarray, target_shape: tuple[int,int,int]) -> np.ndarray:
    """Center-crop if larger; center-pad with zeros if smaller."""
    assert vol.ndim == 3
    inD, inH, inW = vol.shape
    outD, outH, outW = target_shape
    out = np.zeros(target_shape, dtype=vol.dtype)

    def _slices(in_len, out_len):
        if in_len >= out_len:
            s = (in_len - out_len) // 2
            return slice(s, s + out_len), slice(0, out_len)
        else:
            s = (out_len - in_len) // 2
            return slice(0, in_len), slice(s, s + in_len)

    sD_in, sD_out = _slices(inD, outD)
    sH_in, sH_out = _slices(inH, outH)
    sW_in, sW_out = _slices(inW, outW)
    out[sD_out, sH_out, sW_out] = vol[sD_in, sH_in, sW_in]
    return out.astype(np.float32)

def _normalize_image(img: np.ndarray) -> np.ndarray:
    """Robust [0,1] normalize inside nonzero region."""
    if img.size == 0 or img.max() == 0:
        return np.zeros_like(img, dtype=np.float32)
    brain = img[img > 0]
    if brain.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(brain, [1, 99])
    img = np.clip(img, p1, p99)
    m, s = brain.mean(), brain.std()
    if s > 0:
        img = (img - m) / s
    mn, mx = img.min(), img.max()
    return ((img - mn) / (mx - mn + 1e-8)).astype(np.float32)

def _prepare_pair(img: np.ndarray, msk: np.ndarray, target_shape: tuple[int,int,int], resample: bool):
    """
    Make image & mask the SAME shape with identical resample/crop/pad steps.
    - resample=True: map current shape -> target_shape (linear for img, nearest for mask)
    - then enforce exact target via centered crop/pad
    """
    assert img.shape == msk.shape, f"pre-prep mismatch: {img.shape} vs {msk.shape}"

    if resample and img.shape != target_shape:
        zoom_factors = tuple(t / s for t, s in zip(target_shape, img.shape))
        img = zoom(img, zoom_factors, order=1, mode="nearest", prefilter=False)
        msk = zoom(msk, zoom_factors, order=0, mode="nearest", prefilter=False)

    if img.shape != target_shape or msk.shape != target_shape:
        img = _center_crop_or_pad(img, target_shape)
        msk = _center_crop_or_pad(msk, target_shape)

    img = _normalize_image(img)
    msk = (msk > 0.5).astype(np.float32)
    return img.astype(np.float32), msk.astype(np.float32)

class DynamicDataGenerator(tf.keras.utils.Sequence):
    """
    1) Load image & mask (canonical orientation)
    2) (Optional) resample both to target_shape
    3) Center-crop/pad both identically
    4) Normalize image (mask stays binary)
    """
    def __init__(self, pairs, config: DynamicTrainingConfig, is_training=True):
        self.pair_paths   = [(str(img), str(mask)) for img, mask in pairs]
        self.batch_size   = config.BATCH_SIZE
        self.target_shape = tuple(config.INPUT_SHAPE[:-1])  # (D,H,W)
        self.config       = config
        self.is_training  = is_training  # kept for API compatibility (not used)
        self.indexes      = np.arange(len(self.pair_paths))

        # Optional resampling to target shape (default: False; set True in config to enable)
        self.resample_to_target = getattr(config, "RESAMPLE_TO_TARGET", False)

        # Light caching if plenty of RAM
        self._cache_enabled = psutil.virtual_memory().available > 50 * 1024**3
        self._volume_cache  = {} if self._cache_enabled else None

        np.random.shuffle(self.indexes)
        logger.info(
            f"🔧 Dynamic data generator: {len(self.pair_paths)} samples, "
            f"target_shape={self.target_shape}, resample_to_target={self.resample_to_target}, "
            f"cache={'enabled' if self._cache_enabled else 'disabled'}"
        )

    def __len__(self):
        return len(self.pair_paths) // self.batch_size

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)
        if self._cache_enabled:
            self._volume_cache.clear()
        gc.collect()

    def __getitem__(self, index):
        batch_idxs = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch      = [self.pair_paths[i] for i in batch_idxs]

        X = np.zeros((len(batch), *self.target_shape, 1), dtype=np.float32)
        y = np.zeros((len(batch), *self.target_shape, 1), dtype=np.float32)

        for i, (img_p, mask_p) in enumerate(batch):
            img, msk = self._load_and_prepare_pair(img_p, mask_p)
            X[i, ..., 0] = img
            y[i, ..., 0] = msk
        return X, y

    def _load_and_prepare_pair(self, img_path: str, mask_path: str):
        img = _load_image(img_path)
        msk = _load_mask_bin(mask_path)

        # Safety: if raw shapes differ, reconcile mask to image shape first
        if img.shape != msk.shape:
            logger.warning(f"Image/Mask shape mismatch before prep: {img.shape} vs {msk.shape} [{img_path}]")
            msk = _center_crop_or_pad(msk, img.shape)

        img, msk = _prepare_pair(img, msk, self.target_shape, self.resample_to_target)

        # Cheap sanity: if mask has positive voxels but image there is all zeros, warn
        if np.sum(msk) > 0 and float(np.sum(img[msk > 0])) == 0.0:
            logger.warning(f"Mask region has zero image signal after prep: {img_path}")

        return img, msk




# ---------------------------------------------------------------------------
# Losses & metrics (expects model head to output SIGMOID probabilities)
# ---------------------------------------------------------------------------
# Keras 3 first; fall back to TF-Keras if needed
try:
    from keras.saving import register_keras_serializable
except Exception:  # TF 2.x bundled Keras
    from tensorflow.keras.utils import register_keras_serializable  # type: ignore



@register_keras_serializable(package="custom")
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Soft Dice coefficient on PROBABILITIES (0..1).
    - Assumes model head already applies sigmoid; DO NOT add another sigmoid here.
    - Clips y_pred to avoid log/grad extremes with mixed precision.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    intersection = tf.reduce_sum(y_true * y_pred)
    denom = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2.0 * intersection + smooth) / (denom + smooth)


@register_keras_serializable(package="custom")
def dice_loss(y_true, y_pred):
    """1 - Dice coefficient (expects probabilities)."""
    return 1.0 - dice_coefficient(y_true, y_pred)


def _sobel_3d(t):
    """
    3D Sobel gradient via separable 1D kernels using conv3d.
    Expects shape (B, D, H, W, C). Returns gx, gy, gz.
    """
    t = tf.cast(t, tf.float32)
    k = tf.constant([1., 2., 1.], dtype=tf.float32)
    d = tf.constant([-1., 0., 1.], dtype=tf.float32)

    def mk(axis: str):
        if axis == 'x': kx, ky, kz = d, k, k
        elif axis == 'y': kx, ky, kz = k, d, k
        else:            kx, ky, kz = k, k, d  # 'z'
        filt = tf.einsum('i,j,k->ijk', kz, ky, kx)  # (z,y,x)
        filt = filt[:, :, :, tf.newaxis, tf.newaxis] / 32.0
        return tf.cast(filt, tf.float32)

    fx, fy, fz = mk('x'), mk('y'), mk('z')
    gx = tf.nn.conv3d(t, fx, strides=[1,1,1,1,1], padding='SAME')
    gy = tf.nn.conv3d(t, fy, strides=[1,1,1,1,1], padding='SAME')
    gz = tf.nn.conv3d(t, fz, strides=[1,1,1,1,1], padding='SAME')
    return gx, gy, gz


@register_keras_serializable(package="custom")
def boundary_loss(y_true, y_pred):
    """
    Boundary (edge) loss: L1 distance between Sobel gradient magnitudes
    of ground-truth mask and predicted PROBABILITIES.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Keep preds well-behaved under mixed precision
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    gx_t, gy_t, gz_t = _sobel_3d(y_true)
    gx_p, gy_p, gz_p = _sobel_3d(y_pred)

    gtrue = tf.sqrt(gx_t**2 + gy_t**2 + gz_t**2 + 1e-7)
    gpred = tf.sqrt(gx_p**2 + gy_p**2 + gz_p**2 + 1e-7)

    return tf.reduce_mean(tf.abs(gtrue - gpred))


@register_keras_serializable(package="custom")
class CombinedLoss(tf.keras.losses.Loss):
    """
    Serializable Dice+Boundary composite loss.
    Pass weights via CombinedLoss(alpha=..., beta=...) or from your config.
    """
    def __init__(self, alpha=0.4, beta=0.6, name="combined_loss"):
        super().__init__(name=name)
        self.alpha = float(alpha)
        self.beta = float(beta)

    def get_config(self):
        return {"alpha": self.alpha, "beta": self.beta}

    def call(self, y_true, y_pred):
        return self.alpha * dice_loss(y_true, y_pred) + self.beta * boundary_loss(y_true, y_pred)


@register_keras_serializable(package="custom")
def pred_mean(y_true, y_pred):
    """
    Debug metric: average predicted probability. Useful to spot dead outputs.
    """
    return tf.reduce_mean(tf.cast(y_pred, tf.float32))


# --- Known custom objects mapping (layers, losses, metrics) ---
def _custom_objects():
    return {
        "ResidualConvBlock": ResidualConvBlock,
        "VisionMambaBlock": VisionMambaBlock,
        "SAM2Attention": SAM2Attention,
        "CombinedLoss": CombinedLoss,
        "dice_coefficient": dice_coefficient,
        "dice_loss": dice_loss,
        "boundary_loss": boundary_loss,
        "pred_mean": pred_mean,
    }







Visible GPUs: []
Strategy: _DefaultDistributionStrategy


2025-09-23 10:04:24,410 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-09-23 10:04:24,411 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-09-23 10:04:24,412 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
- TensorFlow 2.15.0
- NumPy 1.25.2
- GPU devices: 0


# Re-Test on Held Out validation set from the training

In [7]:
import numpy as np, json, math
from pathlib import Path

# --- tiny helpers (per-sample, not per-batch) ---
def dice_hard_np(y_true, y_bin, eps=1e-6):
    y_true = y_true.astype(np.float32); y_bin = (y_bin > 0).astype(np.float32)
    inter = (y_true*y_bin).sum()
    denom = y_true.sum() + y_bin.sum()
    return (2*inter + eps)/(denom + eps)

def binarize(prob, thr): return (prob >= thr).astype(np.uint8)

def postprocess_3d(mask_bin, min_cc=0, closing_iter=0):
    from scipy.ndimage import label, binary_closing, generate_binary_structure
    out = mask_bin.astype(np.uint8).copy()
    if closing_iter > 0:
        s = generate_binary_structure(3, 1)
        for _ in range(closing_iter): out = binary_closing(out, structure=s).astype(np.uint8)
    if min_cc > 0:
        s = generate_binary_structure(3, 1)
        lbl, n = label(out, structure=s)
        if n > 0:
            sizes = np.bincount(lbl.ravel())
            remove_ids = [i for i in np.where(sizes < min_cc)[0] if i != 0]
            if remove_ids: out[np.isin(lbl, remove_ids)] = 0
    return out

def logit(p, eps=1e-6): p = np.clip(p, eps, 1-eps); return np.log(p/(1-p))
def sigmoid(x): return 1/(1+np.exp(-x))

# --- get probs & truths if not already computed in this session ---
if 'probs' not in globals() or 'truths' not in globals():
    print("Collecting val probs/truths...")
    cfg = DynamicTrainingConfig()
    cfg.INPUT_SHAPE = detect_input_shape(cfg.DATA_DIR) + (1,)
    pairs, lesion_presence = load_generic_dataset(cfg)
    _, val_pairs = create_stratified_splits(pairs, lesion_presence, batch_size=cfg.BATCH_SIZE, test_size=cfg.VALIDATION_SPLIT)
    val_gen = DynamicDataGenerator(val_pairs, cfg, is_training=False)
    import tensorflow as tf
    with tf.distribute.get_strategy().scope():
        model = build_dynamic_model(cfg)
        model.compile(optimizer="adam", loss="binary_crossentropy")
        model.load_weights(str(cfg.checkpoint_path))  # strict load (defs match)
    all_probs, all_truths = [], []
    for i in range(len(val_gen)):
        X, y = val_gen[i]
        p = model.predict(X, verbose=0).astype(np.float32)
        all_probs.append(p); all_truths.append(y.astype(np.float32))
    probs  = np.concatenate(all_probs, axis=0)   # (N,D,H,W,1)
    truths = np.concatenate(all_truths, axis=0)  # (N,D,H,W,1)
print("Using arrays:", probs.shape, truths.shape)

N = truths.shape[0]

# 1) Pure threshold grid (no post-proc), dense
th_grid = np.linspace(0.20, 0.99, 80)
best = (-1.0, None)
for t in th_grid:
    scores = [dice_hard_np(truths[i,...,0], binarize(probs[i,...,0], t)) for i in range(N)]
    m = float(np.mean(scores))
    if m > best[0]: best = (m, float(t))
print(f"[No PP] best mean hard-Dice={best[0]:.4f} @ t={best[1]:.3f}")

# 2) Joint grid: threshold × min_cc × closing
fine_th = np.linspace(max(0.40, best[1]-0.25), min(0.99, best[1]+0.25), 60)
min_cc_grid = [0, 25, 50, 100, 150, 250, 400, 600, 1000]
clos_grid   = [0, 1, 2, 3]
best_joint = (-1.0, None, None, None)  # (mean, t, min_cc, clos)
for t in fine_th:
    for mcc in min_cc_grid:
        for clos in clos_grid:
            scores = []
            for i in range(N):
                mb = binarize(probs[i,...,0], t)
                mb = postprocess_3d(mb, min_cc=mcc, closing_iter=clos)
                scores.append(dice_hard_np(truths[i,...,0], mb))
            m = float(np.mean(scores))
            if m > best_joint[0]:
                best_joint = (m, float(t), int(mcc), int(clos))
print(f"[Grid PP] best mean={best_joint[0]:.4f} @ t={best_joint[1]:.3f}, min_cc={best_joint[2]}, closing={best_joint[3]}")

# 3) Temperature scaling + threshold (probability sharpening; no retraining)
T_grid  = np.linspace(0.6, 1.5, 19)  # T<1 sharpens, T>1 smooths
th_grid2= np.linspace(0.30, 0.95, 66)
best_temp = (-1.0, None, None)
logits = logit(probs[...,0])
for T in T_grid:
    pT = sigmoid(logits / T)[..., None]  # back to shape (...,1)
    for t in th_grid2:
        scores = [dice_hard_np(truths[i,...,0], binarize(pT[i,...,0], t)) for i in range(N)]
        m = float(np.mean(scores))
        if m > best_temp[0]: best_temp = (m, float(T), float(t))
print(f"[Temp] best mean={best_temp[0]:.4f} @ T={best_temp[1]:.2f}, t={best_temp[2]:.3f}")

# 4) Oracle per-case best threshold (upper bound for tuning on this val set)
oracle_ths = np.linspace(0.2, 0.99, 80)
oracle_scores = []
for i in range(N):
    best_i = max(dice_hard_np(truths[i,...,0], binarize(probs[i,...,0], t)) for t in oracle_ths)
    oracle_scores.append(best_i)
print(f"[Oracle per-case] mean upper bound={np.mean(oracle_scores):.4f}, median={np.median(oracle_scores):.4f}")

# Save summary
cfg = DynamicTrainingConfig()  # only to get the callbacks dir
out = cfg.CALLBACKS_DIR / "val_threshold_tuning.json"
summary = {
    "no_postproc": {"best_mean": best[0], "t": best[1]},
    "grid_postproc": {"best_mean": best_joint[0], "t": best_joint[1],
                      "min_cc": best_joint[2], "closing": best_joint[3]},
    "temperature": {"best_mean": best_temp[0], "T": best_temp[1], "t": best_temp[2]},
    "oracle_upper_bound": {"mean": float(np.mean(oracle_scores)),
                           "median": float(np.median(oracle_scores))},
    "N": int(N),
}
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print("Wrote:", out, "\n", json.dumps(summary, indent=2))


Using arrays: (78, 208, 240, 192, 1) (78, 208, 240, 192, 1)
[No PP] best mean hard-Dice=0.3677 @ t=0.980
[Grid PP] best mean=0.3683 @ t=0.990, min_cc=1000, closing=0
[Temp] best mean=0.3677 @ T=1.30, t=0.950


2025-09-24 05:24:44,233 - SmartSOTA_Dynamic - INFO - 📋 DynamicTrainingConfig initialised:
   Data directory: /home/rbielski/Atlas_2/Training_Split/Training_Set
   Batch size: 2
   Warmup epochs: 15
   Minimum LR: 5e-07
   Dice/boundary weights: 0.4:0.6



[Oracle per-case] mean upper bound=0.3746, median=0.3667
Wrote: callbacks/dynamic_production/val_threshold_tuning.json 
 {
  "no_postproc": {
    "best_mean": 0.36768801883379554,
    "t": 0.98
  },
  "grid_postproc": {
    "best_mean": 0.3682579639975669,
    "t": 0.99,
    "min_cc": 1000,
    "closing": 0
  },
  "temperature": {
    "best_mean": 0.36768801883379554,
    "T": 1.3,
    "t": 0.95
  },
  "oracle_upper_bound": {
    "mean": 0.3745790670287047,
    "median": 0.3666939850706996
  },
  "N": 78
}


# Use the best threshold to define the new test function

# Test on actual Test Set 


In [8]:
# === Evaluate on the held-out TEST set with locked-in threshold/post-proc ===
import os, json, logging, warnings
from pathlib import Path
import numpy as np
import tensorflow as tf
import nibabel as nib
from scipy.ndimage import label, binary_closing, generate_binary_structure
from tensorflow.keras import mixed_precision

# --- constants from your tuning run ---
BEST_T   = 0.99
MIN_CC   = 1000
CLOS_IT  = 0
INPUT_SHAPE_TRAIN = (208, 240, 192, 1)   # from training (don’t re-detect for test)

TEST_DIR = Path("/home/rbielski/Atlas_2/Training_Split/Test_set")
CKPT     = Path("callbacks/dynamic_production/best_model_dynamic.weights.h5")
OUT_JSON = Path("callbacks/dynamic_production/test_metrics.json")
SAVE_NIFTI = False  # set True to save predicted masks

# --- quiet NiBabel chatter ---
for name in ("nibabel", "nibabel.global"):
    lg = logging.getLogger(name); lg.setLevel(logging.ERROR); lg.propagate = False
warnings.filterwarnings("ignore",
    message=r".*pixdim\[0\].*qfac.*", category=UserWarning, module=r"nibabel(\..*)?$")

# --- helpers ---
def dice_hard_np(y_true, y_bin, eps=1e-6):
    y_true = y_true.astype(np.float32)
    y_bin  = (y_bin > 0).astype(np.float32)
    inter = (y_true * y_bin).sum()
    denom = y_true.sum() + y_bin.sum()
    return (2*inter + eps) / (denom + eps)

def postprocess_3d(mask_bin, min_cc=200, closing_iter=0):
    out = mask_bin.astype(np.uint8).copy()
    if closing_iter > 0:
        s = generate_binary_structure(3, 1)
        for _ in range(closing_iter):
            out = binary_closing(out, structure=s).astype(np.uint8)
    s = generate_binary_structure(3, 1)
    lbl, n = label(out, structure=s)
    if n > 0:
        sizes = np.bincount(lbl.ravel())
        remove_ids = [i for i in np.where(sizes < min_cc)[0] if i != 0]
        if remove_ids:
            out[np.isin(lbl, remove_ids)] = 0
    return out

def _pair_key(img_path: Path):
    name = img_path.name[:-7] if img_path.name.endswith(".nii.gz") else img_path.name
    return name.replace("_T1w", "")

# --- match training mixed precision & clear session ---
mixed_precision.set_global_policy("mixed_bfloat16")
tf.keras.backend.clear_session()

# --- build a config pointed at TEST set (keep training input shape) ---
cfg = DynamicTrainingConfig()
cfg.DATA_DIR = TEST_DIR
cfg.INPUT_SHAPE = INPUT_SHAPE_TRAIN
# ensure we behave exactly like training preprocessing
if not hasattr(cfg, "RESAMPLE_TO_TARGET"):
    cfg.RESAMPLE_TO_TARGET = True
else:
    cfg.RESAMPLE_TO_TARGET = True  # force on for consistency

# --- load all test pairs (no split) ---
pairs, lesion_presence = load_generic_dataset(cfg)
test_gen = DynamicDataGenerator(pairs, cfg, is_training=False)
print(f"Test samples: {len(pairs)}  |  batches: {len(test_gen)}")

# --- rebuild model exactly like training and load checkpoint ---
with tf.distribute.get_strategy().scope():
    model = build_dynamic_model(cfg)          # your build takes the config object
    model.compile(optimizer="adam", loss="binary_crossentropy")  # metrics not needed for manual scoring

if not CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT}")
try:
    model.load_weights(str(CKPT))
    print(f"Loaded checkpoint: {CKPT}")
except Exception as e:
    print("⚠️ Strict load failed; trying skip_mismatch\n", e)
    model.load_weights(str(CKPT), skip_mismatch=True)
    print("✅ Loaded with skip_mismatch (some layers may not restore).")

# --- run inference and compute per-sample HARD Dice with locked-in settings ---
scores = []
case_ids = []
maybe_preds = []

for i in range(len(test_gen)):
    X, y = test_gen[i]                               # shapes (B,D,H,W,1)
    p = model.predict(X, verbose=0).astype(np.float32)

    for b in range(X.shape[0]):
        prob = p[b, ..., 0]
        gt   = y[b, ..., 0]

        mb = (prob >= BEST_T).astype(np.uint8)
        mb = postprocess_3d(mb, min_cc=MIN_CC, closing_iter=CLOS_IT)

        d = dice_hard_np(gt, mb)
        scores.append(float(d))

        img_path, _ = pairs[i * cfg.BATCH_SIZE + b]
        case_ids.append(_pair_key(img_path))

        if SAVE_NIFTI:
            maybe_preds.append((img_path, mb.astype(np.uint8)))

# --- summarize & save ---
mean_d = float(np.mean(scores))
med_d  = float(np.median(scores))
print(f"\nTEST hard-Dice @ t={BEST_T:.2f}, min_cc={MIN_CC}, closing={CLOS_IT}:")
print(f" - mean={mean_d:.4f}  median={med_d:.4f}  n={len(scores)}")

OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps({
    "threshold": BEST_T,
    "min_cc": MIN_CC,
    "closing_iter": CLOS_IT,
    "mean_hard_dice": mean_d,
    "median_hard_dice": med_d,
    "n": len(scores),
    "cases": [{"id": cid, "hard_dice": float(s)} for cid, s in zip(case_ids, scores)]
}, indent=2))
print("📄 Wrote:", OUT_JSON)

# --- optional mask export (off by default) ---
if SAVE_NIFTI and maybe_preds:
    out_dir = Path("callbacks/dynamic_production/test_preds")
    out_dir.mkdir(parents=True, exist_ok=True)
    for img_path, mb in maybe_preds:
        # Write a NIfTI aligned to the test image header/affine
        img = nib.load(str(img_path))
        nii = nib.Nifti1Image(mb.astype(np.uint8), img.affine, img.header)
        nib.save(nii, str(out_dir / f"{_pair_key(img_path)}_pred.nii.gz"))
    print(f"🧠 Saved {len(maybe_preds)} predicted masks to {out_dir}")


2025-09-24 10:11:30,345 - SmartSOTA_Dynamic - INFO - 📋 DynamicTrainingConfig initialised:
   Data directory: /home/rbielski/Atlas_2/Training_Split/Training_Set
   Batch size: 2
   Warmup epochs: 15
   Minimum LR: 5e-07
   Dice/boundary weights: 0.4:0.6

2025-09-24 10:11:30,347 - SmartSOTA_Dynamic - INFO - 📚 Loading generic dataset (RB pairing rules)...
2025-09-24 10:11:30,348 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=34.00GB | GPU mem tracking failed | Disk: 1428.1GB free
2025-09-24 10:11:30,351 - SmartSOTA_Dynamic - INFO - ✅ Found 131 images under /home/rbielski/Atlas_2/Training_Split/Test_set/Images
2025-09-24 10:11:30,352 - SmartSOTA_Dynamic - INFO - ✅ Found 131 masks under  /home/rbielski/Atlas_2/Training_Split/Test_set/Masks
2025-09-24 10:12:00,197 - SmartSOTA_Dynamic - INFO - 📊 Created 131 image–mask pairs
2025-09-24 10:12:00,197 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-09-24 10:12:00,198 - SmartSOTA_Dynamic - INFO - Memory at dataset_load

Test samples: 131  |  batches: 65


2025-09-24 10:12:00,813 - absl - WARNING - Skipping variable loading for optimizer 'Adam', because it has 1 variables whereas the saved optimizer has 336 variables. 


Loaded checkpoint: callbacks/dynamic_production/best_model_dynamic.weights.h5

TEST hard-Dice @ t=0.99, min_cc=1000, closing=0:
 - mean=0.2418  median=0.0740  n=130
📄 Wrote: callbacks/dynamic_production/test_metrics.json
